In [ ]:
# Select Planet and Folder Structure
from typing import Literal
from pathlib import Path

import numpy as np
import ipywidgets
import matplotlib.pyplot as plt

from PIL import Image
from numpy.typing import NDArray
from matplotlib.gridspec import GridSpec

from mirage_texture_parser import MirageTextureLoader


def list_dir(folder: str | Path) -> list[Path]:
    """Scan a folder and return all folders inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    return [f for f in folder.iterdir() if f.is_dir()]


def list_files(folder: str | Path, extension: str | None = None) -> list[Path]:
    """Scan a folder and return all files inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    output_files = []
    for f in folder.iterdir():
        if f.is_file():
            if extension is not None:
                if f.suffix != extension:
                    continue
            output_files.append(f)
    return output_files


def detect_cubemap_bodies(folder: Path) -> dict[str, Path]:
    """Find body information for values within the folder."""
    bodies = {}

    def iter_search(fpath: Path):
        dirs = list_dir(fpath)
        if "Terrain" in [d.stem for d in dirs]:
            bodies[fpath.stem.rsplit("_", 1)[1]] = fpath / Path("Terrain")
            return
        for d in dirs:
            iter_search(d)

    iter_search(folder)
    return bodies


sol_dir = r"C:\Users\rweld\Documents\Kerbal Space Program 1\KSP Mirage Beta\GameData\Sol-Textures\PluginData"

bodies = {}
for f in list_dir(sol_dir):
    if f.name[0].isdigit():
        if int(f.name.split("_", 1)[0]):
            bodies.update(detect_cubemap_bodies(f))

select_widget = ipywidgets.Select(
    options=list(bodies.keys()),
    description="Select Body:",
    disabled=False,
)

select_widget

In [ ]:
# Define Cube Map Allocations
body_name = select_widget.value


class CubeMapPlanet:
    """Pull the files from Terrain to make a LOD cube map!"""

    def __init__(self, fpath: str | Path):
        self.folder = fpath if isinstance(fpath, Path) else Path(fpath)
        self.num_layers = len(list_dir(self.folder))

    def get_texture(
        self,
        face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"],
        index: int,
        lod: int,
        image_type: Literal["colour", "height", "normal"] = "colour",
    ) -> NDArray:
        """Pull a single texture from the specified face."""
        textures = MirageTextureLoader(self.folder, lod)
        assert textures.colour_idx.num_entries == 6 * 4**lod  # 1, 4, 16 tex per face
        tex_per_face = textures.colour_idx.num_entries // 6
        tex_order = ["Xp", "Xn", "Yp", "Yn", "Zp", "Zn"]

        return textures.get_texture(
            tex_order.index(face) * tex_per_face + index, image_type
        )

    def project_pixel(
        self, lat: float, lon: float
    ) -> tuple[Literal["Xp", "Xn", "Yp", "Yn", "Zp", "Zn"], tuple[int, int]]:
        """Convert a lat / lon value into a pixel coordinate for the speficied face. Pixel location is (row, column) order with range [0, 1]."""

        rlon = lon * np.pi / 180  # To radians
        rlat = lat * np.pi / 180

        x = np.cos(rlon) * np.cos(rlat)
        y = np.sin(rlat)
        z = np.sin(rlon) * np.cos(rlat)

        a = max(np.abs(x), np.abs(y), np.abs(z))
        # Define u, v : -1 < u < 1 maps horizontal pixels and -1 < v < 1 maps vertical
        if a == np.abs(x):
            side = "Xp" if x > 0 else "Xn"  # 0 or 1
            u = -np.sign(x) * z / a
            v = -y / a
        elif a == np.abs(y):
            side = "Yp" if y > 0 else "Yn"  # 2 or 3
            u = x / a
            v = np.sign(y) * z / a
        else:
            side = "Zp" if z > 0 else "Zn"  # 4 or 5
            u = np.sign(z) * x / a
            v = -y / a

        row = (v + 1) / 2  # Vertical position in image - pixel row
        col = (u + 1) / 2  # Horizontal position in image - pixel column
        return side, (row, col)

    def stitch_image(
        self,
        face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"],
        lod: int = 0,
        image_type: Literal["colour", "height", "normal"] = "colour",
    ):
        """Get the combined image from the tiles in the texture."""
        assert lod < self.num_layers
        images = []

        textures = MirageTextureLoader(self.folder, lod)
        assert textures.colour_idx.num_entries == 6 * 4**lod  # 1, 4, 16 tex per face
        tex_per_face = textures.colour_idx.num_entries // 6
        tex_order = ["Xp", "Xn", "Yp", "Yn", "Zp", "Zn"]

        if lod == 0:
            return textures.get_texture(tex_order.index(face), image_type)

        order = [[0, 2], [1, 3]]
        indices = order
        for _ in range(lod - 1):
            size = len(indices) ** 2
            first_half = indices + [[x + size for x in segment] for segment in indices]
            last_half = [[x + 2 * size for x in segment] for segment in first_half]
            indices = [x[0] + x[1] for x in zip(first_half, last_half)]

        indices = [x for xs in indices for x in xs]

        for i in indices:
            im_array = textures.get_texture(
                tex_order.index(face) * tex_per_face + i, image_type
            )
            images.append(im_array)

        lines = []
        tile_count = 2**lod
        for i in range(tile_count):
            # Stitch while keeping padding in place
            imgs = images[tile_count * i : tile_count * (i + 1)]
            for j in range(tile_count):
                start = 0 if j == 0 else 4
                end = 264 if j == tile_count - 1 else 260
                imgs[j] = imgs[j][start:end, :, :]

            start = 0 if i == 0 else 4
            end = 264 if i == tile_count - 1 else 260
            imgs = [x[:, start:end, :] for x in imgs]  # Remove padding on middle layers
            res = np.concatenate(imgs, axis=0)
            lines.append(res)

        return np.concatenate(lines, axis=1)


dd = CubeMapPlanet(bodies[body_name])
data = dd.stitch_image("Yn", 3)

# Display Image
fig = plt.figure(figsize=(8, 6))
gs = GridSpec(3, 4, figure=fig, left=0, right=1, top=1, bottom=0)

layer = 3

xn = fig.add_subplot(gs[1, 0])
xn.imshow(dd.stitch_image("Xn", layer))
xn.set_aspect("equal")

zn = fig.add_subplot(gs[1, 1])
zn.imshow(dd.stitch_image("Zp", layer))
zn.set_aspect("equal")

yp = fig.add_subplot(gs[0, 1])
yp.imshow(dd.stitch_image("Yp", layer))
yp.set_aspect("equal")

yn = fig.add_subplot(gs[2, 1])
yn.imshow(dd.stitch_image("Yn", layer))
yn.set_aspect("equal")

xp = fig.add_subplot(gs[1, 2])
xp.imshow(dd.stitch_image("Xp", layer))
xp.set_aspect("equal")

zp = fig.add_subplot(gs[1, 3])
zp.imshow(dd.stitch_image("Zn", layer))
print(dd.stitch_image("Zp", layer).shape)
zp.set_aspect("equal")

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
for ax in fig.get_axes():
    ax.set_xticks([])
    ax.set_yticks([])

plt.axis("off")
plt.show()

In [ ]:
# Work out cube to rectangle projection
import cv2


def bilinear_interpolate(arr: NDArray, x: float, y: float) -> tuple[int, int, int, int]:
    """
    Interpolate a value at (x, y) in a 2D numpy array arr.
    """
    # 1. Identify surrounding grid indices
    x1, y1 = int(np.floor(x)), int(np.floor(y))
    x2, y2 = x1 + 1, y1 + 1

    # Boundary checks to prevent index errors
    x2 = min(x2, arr.shape[1] - 1)
    y2 = min(y2, arr.shape[0] - 1)

    # 2. Extract the four nearest neighbor values
    p11 = tuple(arr[y1, x1, :])  # Bottom-left
    p12 = tuple(arr[y2, x1, :])  # Top-left
    p21 = tuple(arr[y1, x2, :])  # Bottom-right
    p22 = tuple(arr[y2, x2, :])  # Top-right

    # 3. Calculate relative distances (weights)
    x_diff = x - x1
    y_diff = y - y1

    # 4. Compute weighted average
    # Formula: (1-dx)(1-dy)*p11 + (1-dx)*dy*p21 + dx*(1-dy)*p12 + dx*dy*p22
    res = []
    for channel in range(len(p11)):
        interpolated = (
            (1 - x_diff) * (1 - y_diff) * p11[channel]
            + (1 - x_diff) * y_diff * p21[channel]
            + x_diff * (1 - y_diff) * p12[channel]
            + x_diff * y_diff * p22[channel]
        )
        res.append(interpolated)

    return tuple(res)


def scansat_shading(
    colour_map: NDArray, normal_map: NDArray, normal_axis: int = 2
) -> NDArray:
    """Replicate the SCANsat shading code to see what it does."""
    hls_colours = cv2.cvtColor(colour_map, cv2.COLOR_RGB2HLS)

    opacity = 0.8

    lumOver = normal_map[:, :, normal_axis].astype(np.float32) / 255.0
    lum = hls_colours[:, :, 1].astype(np.float32) / 255.0

    new_lum = np.where(
        lum > 0.5,
        (opacity * (1 - (1 - (2 * (lumOver - 0.5))) * (1 - lum))) + (1 - opacity) * lum,
        (opacity * (2 * lumOver * lum)) + (1 - opacity) * lum,
    )

    hls_colours[:, :, 1] = (new_lum * 255.0).astype(np.int8)
    return cv2.cvtColor(hls_colours, cv2.COLOR_HLS2RGB)


def project_uv(cube: CubeMapPlanet, width: int, height: int) -> NDArray:
    """Cube Map to Equirectangular Projection."""

    w_scale = 360 / width
    h_scale = 180 / height

    lod = int(np.ceil(np.log(width) / np.log(2))) - 10
    print(lod)

    cube_sides: dict[str, NDArray] = {}
    for side in ["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"]:
        colour_tile = cube.stitch_image(side, lod, "colour")  # type: ignore[reportTypeArgument]
        # cube_sides[side] = colour_tile
        normal_tile = cube.stitch_image(side, lod, "normal")  # type: ignore [reportTypeArgument]
        cube_sides[side] = scansat_shading(colour_tile, normal_tile, 1)

    pixel = np.zeros([height, width, cube_sides["Xp"].shape[2]], dtype=np.uint8)
    padding = 4  # Number of padded pixels around each image
    s = cube_sides["Xp"].shape[0] - 2 * padding

    for h_pixel in range(height):
        for w_pixel in range(width):
            lat = 90 - h_pixel * h_scale
            lon = w_pixel * w_scale - 180 + 20

            side, (row, col) = cube.project_pixel(lat, lon)
            # p = bilinear_interpolate(cube_sides[side], p2 + padding, p1 + padding)
            # pixel[h_pixel, w_pixel, :] = p
            row = int(round(s * row))
            col = int(round(s * col))
            pixel[h_pixel, w_pixel, :] = cube_sides[side][
                row + padding, col + padding, :
            ]

    return pixel


proj_data = project_uv(dd, 2048, 1024)
print(f"End data shape {proj_data.shape}")

# Display Image
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.axis("off")
Image.fromarray(proj_data).save("image.png")
plt.imshow(proj_data)
plt.show()

In [ ]:
# Orthographic Map Projection
import glob
import contextlib


def refine_texture_projection(
    y: int | float, x: int | float, lod: int
) -> tuple[int, tuple[float, float]]:
    """Return the required index for a sub-texture in the xy plane. x, y in [0, 1]."""

    right_side = int(x > 0.5)
    bottom_side = int(y > 0.5)
    score = right_side + 2 * bottom_side

    new_x = 2 * (x - 0.5 * right_side)
    new_y = 2 * (y - 0.5 * bottom_side)

    if lod == 1:
        return score, (new_y, new_x)
    elif lod > 1:
        sub_score, (sub_y, sub_x) = refine_texture_projection(new_y, new_x, lod - 1)
        return score * 4 ** (lod - 1) + sub_score, (sub_y, sub_x)
    else:
        return 0, (y, x)  # LOD 0 case


def shaded_texture(
    cube: CubeMapPlanet,
    side: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"],
    index: int,
    lod: int = 0,
) -> NDArray:
    """Return the SCANsat shaded sode."""
    colour_tile = cube.get_texture(side, index, lod, "colour")  # type: ignore[reportTypeArgument]
    normal_tile = cube.get_texture(side, index, lod, "normal")  # type: ignore [reportTypeArgument]
    return scansat_shading(colour_tile, normal_tile, 1)


def orthographic_transform(
    x: int | float, y: int | float, base_lat_rad: int | float, base_lon_rad: int | float
) -> tuple[int | float, int | float] | None:
    """Convert values in range [-1, 1] into latitude / longitude values in radians."""
    p = np.sqrt(x**2 + y**2)
    if p > 1:
        return None  # Skip screen pixels that don't intersect a planet
    elif p == 0:
        return base_lat_rad, base_lon_rad
    else:
        c = np.arcsin(p)

        lat = np.arcsin(
            np.cos(c) * np.sin(base_lat_rad)
            - (y * np.sin(c) * np.cos(base_lat_rad)) / p
        )
        lon = base_lon_rad + np.atan2(
            x * np.sin(c),
            p * np.cos(c) * np.cos(base_lat_rad) + y * np.sin(c) * np.sin(base_lat_rad),
        )
        return lat, lon


def generate_orthographic_map(
    cube: CubeMapPlanet,
    width: int,
    height: int,
    base_lat: int | float,
    base_lon: int | float,
    scale: int | float,
) -> NDArray:
    """Work out how to show a better sphere."""
    base_lat = base_lat * np.pi / 180
    base_lon = base_lon * np.pi / 180

    # Work out what textures need to be loaded
    x_lim = 1 / scale
    y_lim = height / width / scale
    points_list: list[tuple[str, tuple[float, float]]] = []
    combo = [
        (x_lim, y_lim),
        (x_lim, 0),
        (x_lim, -y_lim),
        (0, y_lim),
        (0, 0),
        (0, -y_lim),
        (-x_lim, y_lim),
        (-x_lim, 0),
        (-x_lim, -y_lim),
    ]
    for c in combo:
        x, y = c
        ls = np.sqrt(c[0] ** 2 + c[1] ** 2)
        if ls > 1:
            x /= ls
            y /= ls

        coords = orthographic_transform(x, y, base_lat, base_lon)
        if coords is None:
            continue

        lat, lon = coords
        lat = lat * 180 / np.pi
        lon = lon * 180 / np.pi

        points_list.append(cube.project_pixel(lat, lon))

    lod = 0
    corners = [0, 2, 6, 8]
    while len(corners) > 1:
        base = points_list[corners[0]]
        for corner in corners[1:]:
            check = points_list[corner]
            if base[0] == check[0]:  # Same texture side
                pixel_distance = np.sqrt(
                    (base[1][0] - check[1][0]) ** 2 + (base[1][1] - check[1][1]) ** 2
                )
                axis_pixels = width if base[1][0] == check[1][0] else height
                new_lod = int(
                    np.ceil(np.log(axis_pixels / pixel_distance) / np.log(2) - 8)
                )
                lod = min(max(new_lod, lod), cube.num_layers - 1)
                break
        else:
            corners.pop(0)
            continue
        break

    cube_sides: dict[str, dict[int, NDArray]] = {}

    pixel = np.ones([height, width, 3], dtype=np.uint8) * 255
    padding = 4  # Number of padded pixels around each image
    s = 256

    for h_pixel in range(height):
        for w_pixel in range(width):
            x = (2 * w_pixel / width - 1) / scale
            y = (2 * h_pixel / height - 1) * height / width / scale
            coords = orthographic_transform(x, y, base_lat, base_lon)
            if coords is None:
                continue

            lat, lon = coords
            lat = lat * 180 / np.pi
            lon = lon * 180 / np.pi

            side, (bulk_row, bulk_col) = cube.project_pixel(lat, lon)
            index, (row, col) = refine_texture_projection(bulk_row, bulk_col, lod)
            if side not in cube_sides:
                cube_sides[side] = {}
            if index not in cube_sides[side]:
                cube_sides[side][index] = shaded_texture(cube, side, index, lod)
            row = int(s * row + 0.5)
            col = int(s * col + 0.5)
            pixel[h_pixel, w_pixel, : cube_sides[side][index].shape[2]] = cube_sides[
                side
            ][index][row + padding, col + padding, :]

    return pixel


scales = [2**x for x in np.linspace(0, 8, 100)]
for i, scale in enumerate(scales):
    proj_data = generate_orthographic_map(dd, 512, 512, 31, 31.6, scale)
    Image.fromarray(proj_data).save(f"gif_frame_image_{i}.png")
    print(f"Frame {i}/{len(scales)}: Scale {scale:.2f}.")
# proj_data = generate_orthographic_map(dd, 512, 512, 31, 31.6, 10)

# Display Image
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.axis("off")
Image.fromarray(proj_data).save("image.png")
plt.imshow(proj_data)
plt.show()

In [ ]:
# filepaths
from natsort import natsorted

fp_in = "gif_frame_image_*.png"
fp_out = "animated_image.gif"

# use exit stack to automatically close opened images
with contextlib.ExitStack() as stack:
    # lazily load images
    imgs = (stack.enter_context(Image.open(f)) for f in natsorted(glob.glob(fp_in)))

    # extract  first image from iterator
    img = next(imgs)

    # https://pillow.readthedocs.io/en/stable/handbook/image-file-formats.html#gif
    img.save(
        fp=fp_out, format="GIF", append_images=imgs, save_all=True, duration=100, loop=0
    )